#### Dataset

In [1]:
import pandas as pd
from ydata_profiling import ProfileReport

In [2]:
# Cargar todos los datasets
customers = pd.read_csv("data/olist_customers_dataset.csv")
geolocation = pd.read_csv("data/olist_geolocation_dataset.csv")
order_items = pd.read_csv("data/olist_order_items_dataset.csv")
payments = pd.read_csv("data/olist_order_payments_dataset.csv")
reviews = pd.read_csv("data/olist_order_reviews_dataset.csv")
orders = pd.read_csv("data/olist_orders_dataset.csv")
products = pd.read_csv("data/olist_products_dataset.csv")
sellers = pd.read_csv("data/olist_sellers_dataset.csv")
categories = pd.read_csv("data/product_category_name_translation.csv")


In [3]:
# Unir reviews con orders (por order_id)
df = reviews.merge(orders, on="order_id", how="left")

In [4]:
# Unir con payments (por order_id)
df = df.merge(payments.groupby("order_id").agg({
    "payment_value": "sum",
    "payment_installments": "mean",
    "payment_type": lambda x: x.mode()[0] if not x.mode().empty else "unknown"
}).reset_index(), on="order_id", how="left")

In [5]:
# Unir con order_items (por order_id)
df = df.merge(order_items.groupby("order_id").agg({
    "price": "sum",
    "freight_value": "sum",
    "product_id": lambda x: x.mode()[0] if not x.mode().empty else None,
    "seller_id": lambda x: x.mode()[0] if not x.mode().empty else None,
    "order_item_id": "count"
}).rename(columns={"order_item_id": "num_items"}).reset_index(), on="order_id", how="left")

In [6]:
# Unir con customers (por customer_id)
df = df.merge(customers, on="customer_id", how="left")

In [7]:
# Asegurarse de que no haya duplicado antes del merge
if 'product_id' in df.columns:
    df.drop(columns='product_id', inplace=True)

df = df.merge(order_items[['order_id', 'product_id']], on='order_id', how='left')
df = df.merge(products, on='product_id', how='left')

In [8]:
# Asegurarse de que no haya duplicado antes del merge
if 'seller_id' in df.columns:
    df.drop(columns='seller_id', inplace=True)

df = df.merge(order_items[['order_id', 'seller_id']], on='order_id', how='left')
df = df.merge(sellers, on='seller_id', how='left')

In [9]:
# Unir con traducción de categoría (por product_category_name)
df = df.merge(categories, on="product_category_name", how="left")


In [10]:

# Crear el reporte
#profile = ProfileReport(df, title="Reporte del Dataset Unificado", explorative=True)

# Guardarlo como archivo HTML
#profile.to_file("reporte_dataset_unificado.html")


### Limpieza del dataset

In [11]:
# Renombrar las columnas a conservar
df.rename(columns={
    'freight_value_x': 'freight_value',
    'price_x': 'price',
    'num_items_x': 'num_items',
    'product_category_name_english_x': 'product_category_name_english'
}, inplace=True)

In [12]:
# Eliminar columnas con sufijo _y 
df.drop(columns=[
    'freight_value_y',
    'price_y',
    'num_items_y',
    'product_category_name_english_y'
], errors='ignore', inplace=True)


In [13]:
# Eliminar cualquier versión de product_id y seller_id (x, y)
df.drop(columns=[
    'product_id_x', 'product_id_y',
    'seller_id_x', 'seller_id_y'
], errors='ignore', inplace=True)

In [14]:
# Corregir errores de escritura en nombres de columnas
df.columns = [col.replace('lenght', 'length') for col in df.columns]

In [15]:
# Asegurar que sean fechas
df['order_purchase_timestamp'] = pd.to_datetime(df['order_purchase_timestamp'])
df['order_delivered_customer_date'] = pd.to_datetime(df['order_delivered_customer_date'])
df['order_estimated_delivery_date'] = pd.to_datetime(df['order_estimated_delivery_date'])

# Variables derivadas útiles
df['delivery_days'] = (df['order_delivered_customer_date'] - df['order_purchase_timestamp']).dt.days
df['is_late'] = (df['order_delivered_customer_date'] > df['order_estimated_delivery_date']).astype(int)
df['purchase_dayofweek'] = df['order_purchase_timestamp'].dt.dayofweek

In [16]:
df.drop(columns=[
    'order_id', 'customer_id', 'seller_id', 'product_id', 'review_id',
    'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date',
    'order_delivered_customer_date', 'order_estimated_delivery_date',
    'review_creation_date', 'review_answer_timestamp'
], errors='ignore', inplace=True)


In [17]:
df.head(20)

,review_score,review_comment_title,review_comment_message,order_status,payment_value,payment_installments,payment_type,price,freight_value,num_items,...,product_length_cm,product_height_cm,product_width_cm,seller_zip_code_prefix,seller_city,seller_state,product_category_name_english,delivery_days,is_late,purchase_dayofweek
0,4,NaN,NaN,delivered,397.26,8.0,credit_card,370.00,27.26,2.0,...,30.0,30.0,35.0,14600.0,sao joaquim da barra,SP,sports_leisure,6.0,0,3
1,4,NaN,NaN,delivered,397.26,8.0,credit_card,370.00,27.26,2.0,...,30.0,30.0,35.0,14600.0,sao joaquim da barra,SP,sports_leisure,6.0,0,3
2,4,NaN,NaN,delivered,397.26,8.0,credit_card,370.00,27.26,2.0,...,30.0,30.0,35.0,14600.0,sao joaquim da barra,SP,sports_leisure,6.0,0,3
3,4,NaN,NaN,delivered,397.26,8.0,credit_card,370.00,27.26,2.0,...,30.0,30.0,35.0,14600.0,sao joaquim da barra,SP,sports_leisure,6.0,0,3
4,5,NaN,NaN,delivered,88.09,1.0,credit_card,79.79,8.30,1.0,...,19.0,14.0,14.0,12233.0,sao jose dos campos,SP,computers_accessories,9.0,0,2
5,5,NaN,NaN,delivered,194.12,1.0,credit_card,149.00,45.12,1.0,...,20.0,20.0,20.0,37175.0,ilicinea,MG,computers_accessories,13.0,0,5
6,5,NaN,Recebi bem antes do prazo estipulado.,delivered,222.84,1.0,credit_card,179.99,42.85,1.0,...,20.0,20.0,20.0,37175.0,ilicinea,MG,garden_tools,10.0,0,6
7,5,NaN,Parabéns lojas lannister adorei comprar pela I...,delivered,1333.25,10.0,credit_card,1199.00,134.25,1.0,...,105.0,35.0,50.0,81730.0,curitiba,PR,sports_leisure,18.0,0,5
8,1,NaN,NaN,delivered,462.70,1.0,credit_card,418.70,44.00,4.0,...,46.0,6.0,36.0,13405.0,piracicaba,SP,bed_bath_table,5.0,0,4
9,1,NaN,NaN,delivered,462.70,1.0,credit_card,418.70,44.00,4.0,...,46.0,6.0,36.0,13405.0,piracicaba,SP,bed_bath_table,5.0,0,4


In [18]:
df.describe()

,review_score,payment_value,payment_installments,price,freight_value,num_items,customer_zip_code_prefix,product_name_length,product_description_length,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,seller_zip_code_prefix,delivery_days,is_late,purchase_dayofweek
count,157183.000000,157174.000000,157174.000000,156424.000000,156424.000000,156424.000000,157183.000000,154272.000000,154272.000000,154272.000000,156403.000000,156403.000000,156403.000000,156403.000000,156424.000000,153216.000000,157183.000000,157183.000000
mean,3.841796,248.659196,3.202477,207.103909,41.695346,2.253363,34975.347849,48.556381,776.409737,2.112308,2082.336439,30.293172,16.958664,22.885124,23974.913907,11.816605,0.072724,2.722699
std,1.512097,440.674333,3.003417,412.559946,60.130082,2.466392,30129.790778,10.077206,651.040964,1.647227,3734.639920,16.237753,13.795608,11.682445,27399.841416,9.070626,0.259684,1.952572
min,1.000000,0.000000,0.000000,0.850000,0.000000,1.000000,1003.000000,5.000000,4.000000,1.000000,0.000000,7.000000,2.000000,6.000000,1001.000000,0.000000,0.000000,0.000000
25%,3.000000,77.500000,1.000000,56.990000,15.430000,1.000000,11030.000000,42.000000,341.000000,1.000000,300.000000,18.000000,8.000000,15.000000,6317.000000,6.000000,0.000000,1.000000
50%,5.000000,141.085000,2.000000,109.990000,23.380000,1.000000,23587.000000,51.000000,589.000000,1.000000,700.000000,25.000000,13.000000,20.000000,13484.000000,10.000000,0.000000,3.000000
75%,5.000000,255.600000,5.000000,209.400000,45.462500,2.000000,59151.000000,57.000000,970.000000,3.000000,1750.000000,38.000000,21.000000,30.000000,25645.000000,15.000000,0.000000,4.000000
max,5.000000,13664.080000,24.000000,13440.000000,1794.960000,21.000000,99990.000000,76.000000,3992.000000,20.000000,40425.000000,105.000000,105.000000,118.000000,99730.000000,208.000000,1.000000,6.000000


In [19]:
df.isnull().sum().sort_values(ascending=False)



review_comment_title             137360
review_comment_message            86039
delivery_days                      3967
product_category_name_english      2940
product_photos_qty                 2911
product_description_length         2911
product_name_length                2911
product_category_name              2911
product_weight_g                    780
product_length_cm                   780
product_height_cm                   780
product_width_cm                    780
freight_value                       759
num_items                           759
price                               759
seller_zip_code_prefix              759
seller_state                        759
seller_city                         759
payment_type                          9
payment_installments                  9
payment_value                         9
is_late                               0
review_score                          0
customer_state                        0
customer_city                         0


In [69]:
# Eliminar columnas innecesarias
cols_to_drop = [
    'review_comment_title',
    'review_comment_message',
    'product_category_name',       
    'seller_zip_code_prefix', 
    'customer_city',
    'seller_city' ,   
    'customer_unique_id', 
    'customer_zip_code_prefix'
    ]

df.drop(columns=cols_to_drop, inplace=True, errors='ignore')

In [70]:
# Eliminar filas con valores nulos 
df = df[df['num_items'].notna()]

# Eliminar filas con delivery_days nulo 
df = df[df['delivery_days'].notna()]


In [71]:
# Imputar variables numéricas con mediana
num_median_impute = [
    'product_description_length',
    'product_name_length',
    'product_photos_qty',
    'product_height_cm',
    'product_length_cm',
    'product_width_cm',
    'product_weight_g',
    'freight_value',
    'price'
]
for col in num_median_impute:
    df[col].fillna(df[col].median(), inplace=True)


C:\Users\Ale\AppData\Local\Temp\ipykernel_27732\19968600.py:14: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(df[col].median(), inplace=True)
C:\Users\Ale\AppData\Local\Temp\ipykernel_27732\19968600.py:14: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, wh

In [72]:
# Imputar categóricas con 'missing'
cat_fill = [
    'product_category_name_english',
    'seller_state',
    'payment_type'
]
for col in cat_fill:
    df[col].fillna('unknown', inplace=True)

C:\Users\Ale\AppData\Local\Temp\ipykernel_27732\2004608745.py:8: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna('unknown', inplace=True)


In [73]:
#  Imputar numérica con mediana
df['payment_installments'].fillna(df['payment_installments'].median(), inplace=True)
df['payment_value'].fillna(df['payment_value'].median(), inplace=True)

C:\Users\Ale\AppData\Local\Temp\ipykernel_27732\2576313910.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['payment_installments'].fillna(df['payment_installments'].median(), inplace=True)
C:\Users\Ale\AppData\Local\Temp\ipykernel_27732\2576313910.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values al

In [74]:
df.isnull().sum().sort_values(ascending=False)

review_score                     0
product_photos_qty               0
is_late                          0
delivery_days                    0
product_category_name_english    0
seller_state                     0
product_width_cm                 0
product_height_cm                0
product_length_cm                0
product_weight_g                 0
product_description_length       0
order_status                     0
product_name_length              0
customer_state                   0
num_items                        0
freight_value                    0
price                            0
payment_type                     0
payment_installments             0
payment_value                    0
purchase_dayofweek               0
dtype: int64

In [75]:
df.describe()


,review_score,payment_value,payment_installments,price,freight_value,num_items,product_name_length,product_description_length,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,delivery_days,is_late,purchase_dayofweek
count,153216.000000,153216.000000,153216.000000,153216.000000,153216.000000,153216.000000,153216.000000,153216.000000,153216.000000,153216.000000,153216.000000,153216.000000,153216.000000,153216.000000,153216.000000,153216.000000
mean,3.896277,248.005231,3.204285,206.202801,41.743397,2.257414,48.629595,773.725114,2.098743,2080.577485,30.290420,16.923937,22.874093,11.816605,0.074607,2.724011
std,1.477669,440.917729,3.003239,412.290016,60.457613,2.480592,9.992516,646.394574,1.641492,3728.491147,16.210771,13.775221,11.663666,9.070626,0.262757,1.952354
min,1.000000,9.590000,0.000000,0.850000,0.000000,1.000000,5.000000,4.000000,1.000000,0.000000,7.000000,2.000000,6.000000,0.000000,0.000000,0.000000
25%,3.000000,77.520000,1.000000,56.990000,15.440000,1.000000,42.000000,341.000000,1.000000,300.000000,18.000000,8.000000,15.000000,6.000000,0.000000,1.000000
50%,5.000000,141.230000,2.000000,109.990000,23.380000,1.000000,51.000000,589.000000,1.000000,700.000000,25.000000,13.000000,20.000000,10.000000,0.000000,3.000000
75%,5.000000,254.960000,5.000000,208.000000,45.410000,2.000000,57.000000,961.000000,3.000000,1750.000000,38.000000,21.000000,30.000000,15.000000,0.000000,4.000000
max,5.000000,13664.080000,24.000000,13440.000000,1794.960000,21.000000,76.000000,3992.000000,20.000000,40425.000000,105.000000,105.000000,118.000000,208.000000,1.000000,6.000000


In [77]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder

# Variables categóricas a codificar
categorical_cols = ['order_status', 'payment_type', 'customer_state', 'seller_state', 'product_category_name_english']

# Variables numéricas a escalar
numerical_to_scale = [
    'payment_value', 'payment_installments', 'price', 'freight_value',
    'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm',
    'delivery_days', 'product_description_length', 'product_name_length'
]

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_to_scale),
        ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), categorical_cols)
    ],
    remainder='passthrough'
)


In [78]:
X = df.drop(columns='num_items')  # Features
y = df['num_items']               # Target


In [80]:
X.columns.difference(categorical_cols + numerical_to_scale)


Index(['is_late', 'product_photos_qty', 'purchase_dayofweek', 'review_score'], dtype='object')

In [81]:
X_transformed = preprocessor.fit_transform(X)


In [82]:
# Nombres de columnas escaladas
num_feature_names = preprocessor.named_transformers_['num'].get_feature_names_out(numerical_to_scale)

# Nombres de columnas categóricas codificadas
cat_feature_names = preprocessor.named_transformers_['cat'].get_feature_names_out(categorical_cols)

# Nombres de columnas pasadas sin transformar
passthrough_cols = [col for col in X.columns if col not in categorical_cols + numerical_to_scale]

# Unir todos los nombres
final_column_names = list(num_feature_names) + list(cat_feature_names) + passthrough_cols


In [83]:
import numpy as np

if not isinstance(X_transformed, np.ndarray):
    X_transformed = X_transformed.toarray()
    
X_df = pd.DataFrame(X_transformed, columns=final_column_names)
X_df.head()

,payment_value,payment_installments,price,freight_value,product_weight_g,product_length_cm,product_height_cm,product_width_cm,delivery_days,product_description_length,product_name_length,order_status,payment_type,customer_state,seller_state,product_category_name_english,review_score,product_photos_qty,is_late,purchase_dayofweek
0,0.338510,1.596853,0.397288,-0.239564,-0.209355,-0.017915,0.949248,1.039634,-0.641259,0.130377,-0.663458,1.0,1.0,25.0,21.0,65.0,4.0,1.0,0.0,3.0
1,0.338510,1.596853,0.397288,-0.239564,-0.209355,-0.017915,0.949248,1.039634,-0.641259,0.130377,-0.663458,1.0,1.0,25.0,21.0,65.0,4.0,1.0,0.0,3.0
2,0.338510,1.596853,0.397288,-0.239564,-0.209355,-0.017915,0.949248,1.039634,-0.641259,0.130377,-0.663458,1.0,1.0,25.0,21.0,65.0,4.0,1.0,0.0,3.0
3,0.338510,1.596853,0.397288,-0.239564,-0.209355,-0.017915,0.949248,1.039634,-0.641259,0.130377,-0.663458,1.0,1.0,25.0,21.0,65.0,4.0,1.0,0.0,3.0
4,-0.362688,-0.733972,-0.306612,-0.553173,-0.492313,-0.696479,-0.212261,-0.760835,-0.310520,-0.434295,-0.163082,1.0,1.0,25.0,21.0,15.0,5.0,1.0,0.0,2.0


In [84]:
X_df.describe()

,payment_value,payment_installments,price,freight_value,product_weight_g,product_length_cm,product_height_cm,product_width_cm,delivery_days,product_description_length,product_name_length,order_status,payment_type,customer_state,seller_state,product_category_name_english,review_score,product_photos_qty,is_late,purchase_dayofweek
count,1.532160e+05,1.532160e+05,1.532160e+05,1.532160e+05,1.532160e+05,1.532160e+05,1.532160e+05,1.532160e+05,1.532160e+05,1.532160e+05,1.532160e+05,153216.000000,153216.000000,153216.000000,153216.000000,153216.000000,153216.000000,153216.000000,153216.000000,153216.000000
mean,-4.377822e-17,-1.450617e-16,-5.973130e-17,1.461747e-16,6.455432e-17,-1.024893e-16,1.157526e-16,9.349247e-17,-7.494237e-17,-3.895519e-17,2.949465e-16,0.999941,0.843946,18.823256,18.566684,38.887362,3.896277,2.098743,0.074607,2.724011
std,1.000003e+00,1.000003e+00,1.000003e+00,1.000003e+00,1.000003e+00,1.000003e+00,1.000003e+00,1.000003e+00,1.000003e+00,1.000003e+00,1.000003e+00,0.007664,0.594785,7.023951,4.728280,22.176081,1.477669,1.641492,0.262757,1.952354
min,-5.407267e-01,-1.066947e+00,-4.980801e-01,-6.904595e-01,-5.580231e-01,-1.436730e+00,-1.083394e+00,-1.446728e+00,-1.302737e+00,-1.190801e+00,-4.366242e+00,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,1.000000,0.000000,0.000000
25%,-3.866612e-01,-7.339716e-01,-3.619134e-01,-4.350731e-01,-4.775614e-01,-7.581663e-01,-6.478275e-01,-6.750981e-01,-6.412594e-01,-6.694464e-01,-6.634582e-01,1.000000,1.000000,12.000000,19.000000,15.000000,3.000000,1.000000,0.000000,1.000000
50%,-2.421667e-01,-4.009967e-01,-2.333627e-01,-3.037410e-01,-3.702790e-01,-3.263532e-01,-2.848557e-01,-2.464150e-01,-2.002741e-01,-2.857786e-01,2.372188e-01,1.000000,1.000000,22.000000,21.000000,42.000000,5.000000,1.000000,0.000000,3.000000
75%,1.577345e-02,5.979282e-01,4.359079e-03,6.064770e-02,-8.866283e-02,4.755854e-01,2.958991e-01,6.109512e-01,3.509576e-01,2.897232e-01,8.376702e-01,1.000000,1.000000,25.000000,21.000000,59.000000,5.000000,3.000000,0.000000,4.000000
max,3.042771e+01,6.924453e+00,3.209838e+01,2.899920e+01,1.028420e+01,4.608653e+00,6.393825e+00,8.155774e+00,2.162850e+01,4.978825e+00,2.739099e+00,1.000000,4.000000,26.000000,21.000000,71.000000,5.000000,20.000000,1.000000,6.000000


In [87]:
df_full = X_df.copy()
df_full['num_items'] = y.reset_index(drop=True)


In [88]:
from scipy.stats import zscore


# Calcular Z-score sobre variables numéricas
z_scores = np.abs(zscore(df_full[numerical_to_scale]))
mask = (z_scores < 3).all(axis=1)

# Filtrar sin outliers
df_no_outliers = df_full[mask]

In [ ]:
#df_full = df.drop_duplicates()
#df_no_outliers = df.drop_duplicates()

In [91]:
df_full.to_csv("dataset_full.csv", index=False)
df_no_outliers.to_csv("dataset_no_outliers.csv", index=False)
